In [24]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
# ==========================================
# SUPABASE CONNECTION CONFIG
# ==========================================
os.environ["SUPABASE_DB_PASSWORD"] = "Bonabosssfs01"
SUPABASE_HOST = "aws-1-eu-central-1.pooler.supabase.com"
SUPABASE_PORT = "5432"
SUPABASE_DB = "postgres"
SUPABASE_USER = "postgres.ogkdfmkybqtrsglcizzt"
SUPABASE_PASSWORD = os.getenv("SUPABASE_DB_PASSWORD")

if not SUPABASE_PASSWORD:
    raise ValueError("❌ SUPABASE_DB_PASSWORD is not set in environment variables")

# Connection string
DATABASE_URL = f"postgresql+psycopg2://{SUPABASE_USER}:{SUPABASE_PASSWORD}@{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"

# Engine (VERY IMPORTANT SETTINGS)
engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,   # fixes stale connection issues
    pool_recycle=300      # avoids Supabase killing idle connections
)

print("✅ Engine created successfully")

✅ Engine created successfully


In [9]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("✅ Connection test passed:", result.scalar())

✅ Connection test passed: 1


In [10]:
tables = [
    "dim_calendar", "dim_stores", "dim_products", "dim_policy_regimes",
    "fact_sales", "fact_inventory", "fact_weather",
    "fact_promotions", "fact_customer_activity",
    "fact_store_operations", "fact_restriction_events"
]

for table in tables:
    df = pd.read_sql(f"SELECT COUNT(*) AS rows FROM core.{table}", engine)
    print(table, ":", f"{df['rows'][0]:,}")

dim_calendar : 1,943
dim_stores : 7
dim_products : 120
dim_policy_regimes : 4
fact_sales : 1,632,120
fact_inventory : 1,632,120
fact_weather : 13,601
fact_promotions : 380
fact_customer_activity : 13,601
fact_store_operations : 13,601
fact_restriction_events : 22,002


In [11]:
for table in [
    "dim_calendar",
    "fact_sales",
    "fact_inventory",
    "fact_weather",
    "fact_customer_activity",
    "fact_store_operations",
    "fact_restriction_events"
]:
    df = pd.read_sql(f"""
        SELECT MIN(date) AS min_date, MAX(date) AS max_date, COUNT(*) AS rows
        FROM core.{table}
    """, engine)
    print(table)
    print(df)
    print()

dim_calendar
    min_date   max_date  rows
0 2021-01-01 2026-04-27  1943

fact_sales
    min_date   max_date     rows
0 2021-01-01 2026-04-27  1632120

fact_inventory
    min_date   max_date     rows
0 2021-01-01 2026-04-27  1632120

fact_weather
    min_date   max_date   rows
0 2021-01-01 2026-04-27  13601

fact_customer_activity
    min_date   max_date   rows
0 2021-01-01 2026-04-27  13601

fact_store_operations
    min_date   max_date   rows
0 2021-01-01 2026-04-27  13601

fact_restriction_events
    min_date   max_date   rows
0 2021-01-01 2026-04-27  22002



In [12]:
df_check = pd.read_sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT date) AS unique_dates,
        COUNT(DISTINCT store_id) AS unique_stores,
        COUNT(DISTINCT product_id) AS unique_products,
        SUM(units_sold) AS total_units_sold,
        SUM(revenue) AS total_revenue,
        AVG(revenue) AS avg_row_revenue,
        MAX(revenue) AS max_row_revenue
    FROM core.fact_sales
""", engine)

df_check

,rows,unique_dates,unique_stores,unique_products,total_units_sold,total_revenue,avg_row_revenue,max_row_revenue
0,1632120,1943,7,120,3521586.0,1.890468e+11,115829.009075,2344070.0


In [16]:
print(df_check)

      rows  unique_dates  unique_stores  unique_products  total_units_sold  \
0  1632120          1943              7              120         3521586.0   

   total_revenue  avg_row_revenue  max_row_revenue  
0   1.890468e+11    115829.009075        2344070.0  


In [13]:
monthly_revenue = pd.read_sql("""
    SELECT
        DATE_TRUNC('month', date)::date AS month,
        SUM(revenue) AS total_revenue,
        SUM(units_sold) AS total_units_sold
    FROM core.fact_sales
    GROUP BY 1
    ORDER BY 1
""", engine)

monthly_revenue.head()

,month,total_revenue,total_units_sold
0,2021-01-01,3.192953e+09,57192.0
1,2021-02-01,2.785990e+09,49626.0
2,2021-03-01,2.963566e+09,54775.0
3,2021-04-01,2.858082e+09,53679.0
4,2021-05-01,2.988947e+09,56414.0


In [14]:
monthly_revenue.shape

(64, 3)

In [21]:
pd.set_option('display.max_rows', None)
monthly_revenue

,month,total_revenue,total_units_sold
0,2021-01-01,3.192953e+09,57192.0
1,2021-02-01,2.785990e+09,49626.0
2,2021-03-01,2.963566e+09,54775.0
3,2021-04-01,2.858082e+09,53679.0
4,2021-05-01,2.988947e+09,56414.0
5,2021-06-01,2.806938e+09,53063.0
6,2021-07-01,2.890091e+09,55117.0
7,2021-08-01,2.926931e+09,55064.0
8,2021-09-01,2.946596e+09,54750.0
9,2021-10-01,2.948996e+09,55254.0


---
## 6. Peak Revenue & Units — Year / Month Charts